In [ ]:
import polars as pl
from pathlib import Path

# Configuración de rutas
BASE_DIR = Path('C:/Users/marce/F1-data-project/project/')
carreras = ["australia_2026", "china_2026", "japan_2026"]

print("🏎️ Iniciando Generación de MasterV2 (Telemetría Agregada)...")

for c in carreras:
    print(f"\n--- Procesando {c} ---")
    
    car_csv_path = BASE_DIR / "data" / "raw" / c / "car_data.csv"
    # Leemos el master original para no dañarlo
    master_original_path = BASE_DIR / "data" / "processed" / f"{c}_master.parquet"
    # Definimos el nuevo nombre de salida
    master_v2_path = BASE_DIR / "data" / "processed" / f"{c}_masterv2.parquet"
    
    if not car_csv_path.exists() or not master_original_path.exists():
        print(f"⚠️ Faltan archivos para {c}. Verifica car_data.csv y el master original.")
        continue
        
    # 1. Carga y Limpieza de Zonas Horarias (car_data)
    df_car = pl.read_csv(car_csv_path)
    if df_car["date"].dtype in [pl.Utf8, pl.String]:
        df_car = df_car.with_columns(
            pl.col("date").str.to_datetime(time_zone="UTC", strict=False).dt.replace_time_zone(None)
        )
    else:
        df_car = df_car.with_columns(pl.col("date").dt.replace_time_zone(None))
    df_car = df_car.sort("date")
    
    # 2. Carga y Limpieza de Zonas Horarias (master original)
    df_master = pl.read_parquet(master_original_path)
    if "date_start" in df_master.columns:
        if df_master["date_start"].dtype in [pl.Utf8, pl.String]:
            df_master = df_master.with_columns(
                pl.col("date_start").str.to_datetime(time_zone="UTC", strict=False).dt.replace_time_zone(None)
            )
        else:
            df_master = df_master.with_columns(pl.col("date_start").dt.replace_time_zone(None))
        df_master = df_master.sort("date_start")
    
    # 3. Asignación de vuelta a cada milisegundo (Join Asof)
    df_laps_ref = df_master.select(["driver_number", "lap_number", "date_start"]).drop_nulls().sort("date_start")
    
    df_car_mapped = df_car.join_asof(
        df_laps_ref,
        left_on="date",
        right_on="date_start",
        by="driver_number",
        strategy="backward"
    )
    
    # 4. Agregación Táctica (Reducir milisegundos a métricas de vuelta)
    print(f"   Calculando métricas de pedales para {c}...")
    df_car_agg = df_car_mapped.group_by(["driver_number", "lap_number"]).agg([
        pl.col("throttle").mean().alias("throttle_mean_lap"),
        pl.col("throttle").std().alias("throttle_std_lap"),
        pl.col("brake").max().alias("brake_max_lap"),
        pl.col("brake").max().alias("brake_max_lap"),
        pl.col("rpm").max().alias("rpm_max_lap"),
        pl.col("n_gear").max().alias("n_gear_max_lap"),
        pl.col("drs").max().alias("drs_max_lap")
    ])
    
    # 5. Fusión Final y Creación del MasterV2
    df_master_v2 = df_master.join(
        df_car_agg,
        on=["driver_number", "lap_number"],
        how="left"
    )
    
    # Asegurar columna de carrera
    df_master_v2 = df_master_v2.with_columns(pl.lit(c).alias("race_name"))
    
    # GUARDAR COMO V2
    df_master_v2.write_parquet(master_v2_path, compression="snappy")
    print(f"✅ ¡MasterV2 creado!: {master_v2_path.name}")
    
print("\n🏁 Proceso finalizado. Ahora puedes usar los archivos _masterv2.parquet para el Feature Engineering.")

🏎️ Iniciando Generación de MasterV2 (Telemetría Agregada)...

--- Procesando australia_2026 ---
   Calculando métricas de pedales para australia_2026...
✅ ¡MasterV2 creado!: australia_2026_masterv2.parquet

--- Procesando china_2026 ---


C:\Users\marce\AppData\Local\Temp\ipykernel_16832\3925384175.py:47: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df_car_mapped = df_car.join_asof(


   Calculando métricas de pedales para china_2026...
✅ ¡MasterV2 creado!: china_2026_masterv2.parquet

--- Procesando japan_2026 ---
   Calculando métricas de pedales para japan_2026...
✅ ¡MasterV2 creado!: japan_2026_masterv2.parquet

🏁 Proceso finalizado. Ahora puedes usar los archivos _masterv2.parquet para el Feature Engineering.


In [4]:
df_aus = pl.read_parquet('C:/Users/marce/F1-data-project/project/data/processed/australia_2026_masterv2.parquet')

In [5]:
df_aus.columns

['meeting_key',
 'session_key',
 'driver_number',
 'lap_number',
 'date_start',
 'duration_sector_1',
 'duration_sector_2',
 'duration_sector_3',
 'i1_speed',
 'i2_speed',
 'is_pit_out_lap',
 'lap_duration',
 'segments_sector_1',
 'segments_sector_2',
 'segments_sector_3',
 'st_speed',
 'position',
 'compound',
 'stint_number',
 'tyre_age',
 'pit_duration',
 'is_pit_lap',
 'throttle_mean_lap',
 'throttle_std_lap',
 'brake_max_lap',
 'rpm_max_lap',
 'n_gear_max_lap',
 'drs_max_lap',
 'race_name']

In [6]:
df_aus.head()

meeting_key,session_key,driver_number,lap_number,date_start,duration_sector_1,duration_sector_2,duration_sector_3,i1_speed,i2_speed,is_pit_out_lap,lap_duration,segments_sector_1,segments_sector_2,segments_sector_3,st_speed,position,compound,stint_number,tyre_age,pit_duration,is_pit_lap,throttle_mean_lap,throttle_std_lap,brake_max_lap,rpm_max_lap,n_gear_max_lap,drs_max_lap,race_name
i64,i64,i64,i64,datetime[μs],f64,f64,f64,f64,f64,bool,f64,str,str,str,f64,i64,str,f64,f64,f64,i64,f64,f64,i64,i64,i64,str,str
1279,11234,6,2,2026-03-08 04:06:28.913,30.094,18.602,36.86,238.0,276.0,false,85.556,"""[None, 2051, 2048, 2049, 2049,…","""[2048, 2048, 2048, 2048, 2048]""","""[2049, 2048, 2048, 2049, 2048,…",294.0,3,"""MEDIUM""",1.0,1.0,0.0,0,66.350769,40.947648,104,11886,8,null,"""australia_2026"""
1279,11234,41,2,2026-03-08 04:06:30.170,30.171,17.966,37.36,255.0,304.0,false,85.497,"""[None, 2049, 2049, 2048, 2049,…","""[2048, 2049, 2048, 2048, 2048]""","""[2048, 2049, 2048, 2049, 2049,…",289.0,2,"""MEDIUM""",1.0,1.0,0.0,0,null,null,null,null,null,null,"""australia_2026"""
1279,11234,1,2,2026-03-08 04:06:30.675,30.541,18.252,38.07,245.0,311.0,false,86.863,"""[None, 2049, 2049, 2048, 2049,…","""[2049, 2048, 2048, 2048, 2048]""","""[2049, 2048, 2048, 2049, 2049,…",273.0,8,"""MEDIUM""",1.0,1.0,0.0,0,60.924699,44.222423,104,12704,8,null,"""australia_2026"""
1279,11234,12,2,2026-03-08 04:06:30.967,30.586,18.042,36.835,256.0,304.0,false,85.463,"""[None, 2048, 2048, 2048, 2048,…","""[2049, 2048, 2048, 2048, 2049]""","""[2048, 2049, 2049, 2049, 2049,…",276.0,1,"""UNKNOWN""",1.0,1.0,0.0,0,null,null,null,null,null,null,"""australia_2026"""
1279,11234,31,2,2026-03-08 04:06:31.392,30.865,17.978,37.724,null,308.0,false,86.567,"""[None, 2048, 2049, 2048, 2048,…","""[2049, 2048, 2048, 2048, 2049]""","""[2049, 2049, 2048, 2049, 2049,…",267.0,7,"""UNKNOWN""",1.0,1.0,0.0,0,null,null,null,null,null,null,"""australia_2026"""
